# Packages import

In [ ]:
from bs4 import BeautifulSoup
import json
import pandas as pd
import re
import requests
import time
import unicodedata
import urllib.parse

# Functions definition

Tout au long du Notebook il y a des cellules pour évaluer les fonctions développées. Pour les exécuter, enlever les # de commentaires.

## Normaliser_texte

In [52]:
def normaliser_texte(texte):
    
    '''
    Prépare et uniformise une chaîne de caractères pour permettre une comparaison textuelle fiable.
    Notamment : gestion des accents, uniformisation de la casse, nettoyage des espaces.

    Args:
        texte (str): La chaîne de caractères brute à traiter (titre ou nom d'auteur).

    Returns:
        str: Le texte simplifié, sans accents, en minuscules et épuré.
    '''
    
    if not isinstance(texte, str): return ""
    nfkd = unicodedata.normalize('NFKD', texte)
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).lower().strip()

In [ ]:
#normaliser_texte('Monsieur le Préfet ')

'monsieur le prefet'

## Livraddict 2 versions

In [53]:
def search_livraddict(titre, auteur):

    '''
    Recherche et extrait le résumé complet d'un livre sur le site Livraddict avec le titre du livre et le nom de l'auteur.

    Args:
        titre (str): Le titre du livre.
        auteur (str): Le nom de l'auteur ou des auteurs.

    Returns:
        str: Le résumé complet du livre, ou un message d'erreur si l'extraction a échoué.
    '''

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0'
    }
    
    # NETTOYAGE DE L'AUTEUR : on ne garde que le premier auteur 
    if not isinstance(auteur, str): return "Auteur inconnu"
    auteur_clean = re.split(r'\s+(?:et|and|&|,)\s+', auteur, flags=re.IGNORECASE)[0].strip()
    if '&' in auteur_clean: auteur_clean = auteur_clean.split('&')[0].strip()

    try:
        # ÉTAPE 1 : TROUVER LA PAGE AUTEUR 
        url_recherche = f"https://www.livraddict.com/search.php?t={urllib.parse.quote(auteur_clean)}"
        res = requests.get(url_recherche, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # Ciblage de l'onglet spécifique identifié sur l'inspecteur d'éléments
        tab_auteurs = soup.find('div', id='tab_auteurs')
        lien_auteur = None
        if tab_auteurs:
            lien_tag = tab_auteurs.find('a')
            if lien_tag: lien_auteur = lien_tag.get('href')
        
        if not lien_auteur: return f"Auteur '{auteur_clean}' non trouvé"
        url_auteur = lien_auteur if lien_auteur.startswith('http') else f"https://www.livraddict.com{lien_auteur}"

        # ÉTAPE 2 : TROUVER LE LIVRE DANS LA BIBLIOGRAPHIE
        res_auteur = requests.get(url_auteur, headers=headers)
        soup_auteur = BeautifulSoup(res_auteur.text, 'html.parser')
        
        # Ciblage du tableau des œuvres de l'auteur
        table_biblio = soup_auteur.find('table', id='bookAuthor')
        if not table_biblio: return "Tableau 'bookAuthor' introuvable"

        url_livre_trouve = None
        titre_cible = normaliser_texte(titre) 
        
        for row in table_biblio.find_all('tr'):
            lien = row.find('a')
            if lien and '/biblio/livre/' in lien.get('href', ''):
                if titre_cible in normaliser_texte(lien.get_text()):
                    url_livre_trouve = lien['href']
                    break
        
        if not url_livre_trouve: return f"Livre '{titre}' non trouvé dans la liste"
        url_livre_trouve = url_livre_trouve if url_livre_trouve.startswith('http') else f"https://www.livraddict.com{url_livre_trouve}"

        # ÉTAPE 3 : EXTRACTION DU RÉSUMÉ COMPLET
        res_livre = requests.get(url_livre_trouve, headers=headers)
        html_content = res_livre.text
        soup_livre = BeautifulSoup(html_content, 'html.parser')

        # STRATÉGIE A - Regex JSON
        match = re.search(r'"description":\s*"(.*?)(?<!\\)"\s*,', html_content, re.DOTALL)
        if match:
            try:
                return json.loads(f'"{match.group(1)}"')
            except:
                return match.group(1)

        # STRATÉGIE B - Visuelle
        div_visuelle = soup_livre.find('div', class_='synopsis_content')
        if div_visuelle:
            return div_visuelle.get_text(separator='\n', strip=True)

        # STRATÉGIE C - Meta
        meta = soup_livre.find('meta', attrs={'name': 'description'})
        if meta: return "[Partiel] " + meta['content']

        return "Résumé introuvable."

    except Exception as e:
        return f"Erreur technique : {str(e)}"

In [ ]:
#search_livraddict('Fictions', 'Borges Jorge Luis')

"Sans doute y a-t-il du dilettantisme dans ces Fictions, jeux de l'esprit et exercices de style fort ingénieux. Pourtant, le pluriel signale d'emblée qu'il s'agit d'une réflexion sur la richesse foisonnante de l'imagination. Au nombre de dix-huit, ces contes fantastiques révèlent, chacun à sa manière, une ambition totalisante qui s'exprime à travers de nombreux personnages au projet démiurgique ou encore à travers La Bibliothèque de Babel, qui prétend contenir l'ensemble des livres, existants ou non. \r\nLa multitude d'univers parallèles et d'effets de miroir engendrent un &quot;délire circulaire&quot; vertigineux, une interrogation sur la relativité du temps et de l'espace. Dans quelle dimension sommes-nous ? Qui est ce &quot;je&quot; qui raconte l'invasion de la cité dans La Loterie de Babylone ? En mettant en vis-à-vis le Quichotte de Ménard et celui de Cervantès, lit-on la même chose ou bien la décision de redire suffit-elle à rendre la redite impossible ? \r\n\r\nIl n'est pas cert

In [55]:
def search_livraddict_titre_only(titre):

    """
    Recherche et extrait le résumé complet d'un livre sur le site Livraddict avec uniquement le titre du livre.

    Args:
        titre (str): Le titre du livre.

    Returns:
        str: Le résumé complet du livre, ou un message d'erreur si l'extraction a échoué.
    """
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0'}
    
    try:
        # ETAPE 1 : Recherche directe du titre
        url_recherche = f"https://www.livraddict.com/search.php?t={urllib.parse.quote(titre)}"
        print(f"   -> Recherche titre direct : {url_recherche}")
        
        res = requests.get(url_recherche, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # ETAPE 2 : Ciblage de l'onglet LIVRES
        tab_livres = soup.find('div', id='tab_livres')
        
        url_livre_trouve = None
        
        if tab_livres:
            # On récupère tous les résultats de l'onglet livres
            liens = tab_livres.find_all('a')
            
            titre_cible = normaliser_texte(titre)
            
            for lien in liens:
                href = lien.get('href', '')
                if '/biblio/livre/' in href:
                    # On compare le texte du lien avec le titre cherché
                    if titre_cible in normaliser_texte(lien.get_text()):
                        url_livre_trouve = href
                        print(f"   -> Livre trouvé : {lien.get_text(strip=True)}")
                        break
            
            # Si pas de correspondance exacte, on tente le premier lien livre trouvé
            if not url_livre_trouve and liens:
                for lien in liens:
                    if '/biblio/livre/' in lien.get('href', ''):
                        url_livre_trouve = lien['href']
                        print(f"   -> (Fallback) Premier livre trouvé : {lien.get_text(strip=True)}")
                        break
        
        if not url_livre_trouve:
            return "Livre non trouvé dans les résultats de recherche."

        if not url_livre_trouve.startswith('http'):
            url_livre_trouve = f"https://www.livraddict.com{url_livre_trouve}"

        # ETAPE 3 : Extraction du Résumé 
        res_livre = requests.get(url_livre_trouve, headers=headers)
        html_content = res_livre.text
        soup_livre = BeautifulSoup(html_content, 'html.parser')

        # Stratégie A - Regex JSON
        match = re.search(r'"description":\s*"(.*?)(?<!\\)"\s*,', html_content, re.DOTALL)
        if match:
            try:
                return json.loads(f'"{match.group(1)}"')
            except:
                return match.group(1)

        # Stratégie B - Visuelle
        div_visuelle = soup_livre.find('div', class_='synopsis_content')
        if div_visuelle:
            return div_visuelle.get_text(separator='\n', strip=True)

        # Stratégie C - Meta
        meta = soup_livre.find('meta', attrs={'name': 'description'})
        if meta: return "[Partiel] " + meta['content']

        return "Résumé non trouvé sur la page."

    except Exception as e:
        return f"Erreur recherche titre : {str(e)}"

In [ ]:
#search_livraddict_titre_only("2084 : La fin du monde")

   -> Recherche titre direct : https://www.livraddict.com/search.php?t=2084%20%3A%20La%20fin%20du%20monde
   -> Livre trouvé : 2084 : La fin du monde


'L’Abistan, immense empire, tire son nom du prophète Abi, «délégué» de Yölah sur terre. Son système est fondé sur l’amnésie et la soumission au dieu unique. Toute pensée personnelle est bannie, un système de surveillance omniprésent permet de connaître les idées et les actes déviants. Officiellement, le peuple unanime vit dans le bonheur de la foi sans questions. \r\nLe personnage central, Ati, met en doute les certitudes imposées. Il se lance dans une enquête sur l’existence d’un peuple de renégats, qui vit dans des ghettos, sans le recours de la religion… \r\nBoualem Sansal s’est imposé comme une des voix majeures de la littérature contemporaine. Au fil d’un récit débridé, plein d’innocence goguenarde, d’inventions cocasses ou inquiétantes, il s’inscrit dans la filiation d’Orwell pour brocarder les dérives et l’hypocrisie du radicalisme religieux qui menace les démocraties.'

## Other search google books and goodreads

In [ ]:
def search_google_books_api(titre, auteur):

    """
    Recherche et extrait le résumé complet d'un livre sur le site Google Books via l'API publique avec le titre du livre et le nom de l'auteur.

    Args:
        titre (str): Le titre du livre.
        auteur (str): Le nom de l'auteur ou des auteurs.

    Returns:
        str: Le résumé complet du livre, ou un message d'erreur si l'extraction a échoué.
     """

    try:
        query = f"intitle:{titre}+inauthor:{auteur}"
        url = f"https://www.googleapis.com/books/v1/volumes?q={query}&langRestrict=fr&maxResults=1"
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            if "items" in data and len(data["items"]) > 0:
                volume_info = data["items"][0]["volumeInfo"]
                if "description" in volume_info:
                    return volume_info["description"]
    except Exception:
        pass 
    return "Résumé non trouvé sur la page Google Books."

In [ ]:
#search_google_books_api("Ellana", "Pierre Bottero")

'Seule survivante d’un groupe de pionniers après l’attaque de leur caravane, une fi llette est recueillie par un peuple sylvestre et grandit à l’écart des hommes. À l’adolescence, elle décide de partir en quête de ses origines. Sous le nom d’Ellana, elle croise alors le plus grand maître marchombre, Jilano Alhuïn, qui la prend pour élève et l’initie aux secrets de sa guilde. Un apprentissage semé de rencontres et de dangers...Le Pacte des Marchombres invite le lecteur à pénétrer dans les arcanes d’une guilde aux pouvoirs extraordinaires, et à suivre le destin d’Ellana Caldin, héroïne prodigieuse par sa psychologie, ses exploits physiques et son insatiable goût de la liberté.'

In [67]:
def search_goodreads(titre, auteur):

    """
    Cherche un livre sur Goodreads par son titre, valide par l'auteur.


    Args:
        titre (str): Le titre du livre.
        auteur (str): Le nom de l'auteur ou des auteurs.

    Returns:
        str: Le résumé complet du livre, ou un message d'erreur si l'extraction a échoué.
    """

    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',}
    
    query = urllib.parse.quote(titre)
    url_recherche = f"https://www.goodreads.com/search?q={query}"
    
    try:
        # ÉTAPE 1 : RECHERCHE PAR TITRE
        res = requests.get(url_recherche, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        items = soup.find_all('tr', itemtype="http://schema.org/Book")
        url_livre_final = None
        auteur_cible = normaliser_texte(auteur)
        
        for item in items:
            author_tag = item.find('a', class_='authorName')
            if author_tag:
                auteur_trouve = normaliser_texte(author_tag.get_text())
                if auteur_cible in auteur_trouve or auteur_trouve in auteur_cible:
                    link_tag = item.find('a', class_='bookTitle')
                    if link_tag:
                        url_livre_final = "https://www.goodreads.com" + link_tag.get('href')
                        print(f"   -> Match validé sur Goodreads : {author_tag.get_text()} - {link_tag.get_text(strip=True)}")
                        break
        
        if not url_livre_final:
            return "Livre non trouvé avec cet auteur sur Goodreads"

        # ÉTAPE 2 : EXTRACTION DU RÉSUMÉ 
        res_livre = requests.get(url_livre_final, headers=headers)
        soup_livre = BeautifulSoup(res_livre.text, 'html.parser')
        
        # Sécurité 
        if soup_livre.title and "Just a moment" in soup_livre.title.string:
            return "Erreur : Goodreads a bloqué l'accès à la page du livre (Captcha)."

        # STRATÉGIE A : Le JSON-LD 
        script_json = soup_livre.find('script', type='application/ld+json')
        if script_json and script_json.string:
            try:
                data = json.loads(script_json.string)
                if 'description' in data:
                    # Le JSON contient parfois des balises HTML <br>, on les nettoie
                    texte_propre = BeautifulSoup(data['description'], "html.parser").get_text(separator='\n')
                    return texte_propre
            except:
                pass 

        # STRATÉGIE B : Meta
        meta_desc = soup_livre.find('meta', attrs={'name': 'description'})
        if meta_desc:
            content = meta_desc.get('content', '')
            if len(content) > 20:
                # Cette Regex supprime tout ce qui précède "community for readers." 
                # même s'il y a des espaces, sauts de ligne ou textes variables avant.
                content = re.sub(r".*?community for readers\.\s*", "", content, flags=re.IGNORECASE | re.DOTALL)
                return content.strip()

        # STRATÉGIE C : Les balises <div> spécifiques 
        desc_div = soup_livre.find('div', attrs={'data-testid': 'description'}) or soup_livre.find('div', id='description')
        if desc_div:
            return desc_div.get_text(separator='\n', strip=True)

        return "Résumé introuvable sur la page (Structure HTML non reconnue)"

    except Exception as e:
        return f"Erreur technique Goodreads : {str(e)}"

In [ ]:
#search_goodreads("Magnolia Parks", "Jessa Hastings")

   -> Match validé sur Goodreads : Jessa Hastings - Magnolia Parks (Magnolia Parks Universe, #1)


'Read 54.1k reviews from the world’s largest community \n    for readers. She is a beautiful, affluent, self-involved and mildly neurotic London socialite. He is …'

## Obtention du résumé

In [75]:
def obtenir_resume_intelligent(titre, auteur):
    
    """
    Fonction principale qui orchestre la recherche en cascade.
    Ordre de priorité : Livraddict -> Google Books -> Goodreads.
    Utilise la fonction de recherche avec le titre uniquement pour Livraddict lorsque l'auteur est un collectif.

    Args:
        titre (str): Le titre du livre.
        auteur (str): Le nom de l'auteur ou des auteurs.

    Returns:
        str: Le résumé complet du livre, ou un message d'erreur si l'extraction a échoué.
    """
    auteur_normalise = normaliser_texte(auteur)

    print(f"Recherche pour : {titre} ...")

    if "collectif" in auteur_normalise:
        print("   -> Auteur collectif détecté")
        resume = search_livraddict_titre_only(titre)
        if resume:
            return f"{resume}"
        else:
            print("   -> Aucun résumé trouvé")

    # ÉTAPE 1 : Livraddict 
    resume = search_livraddict(titre, auteur)
    if resume and "non trouvé" not in resume.lower() and "Erreur" not in resume:
        return f"{resume}"
    
    # ÉTAPE 2 : Google Books API 
    resume = search_google_books_api(titre, auteur)
    if resume:
        return f"{resume}"
    
    # ÉTAPE 3 : Goodreads 
    resume = search_goodreads(titre, auteur)
    if resume and "non trouvé" not in resume.lower():
        return f"{resume}"

    return "Aucun résumé trouvé sur les différentes plateformes."

In [ ]:
#obtenir_resume_intelligent("Fictions", "Jorge Luis Borges")

Recherche pour : Fictions ...


"[Source: Livraddict] Sans doute y a-t-il du dilettantisme dans ces Fictions, jeux de l'esprit et exercices de style fort ingénieux. Pourtant, le pluriel signale d'emblée qu'il s'agit d'une réflexion sur la richesse foisonnante de l'imagination. Au nombre de dix-huit, ces contes fantastiques révèlent, chacun à sa manière, une ambition totalisante qui s'exprime à travers de nombreux personnages au projet démiurgique ou encore à travers La Bibliothèque de Babel, qui prétend contenir l'ensemble des livres, existants ou non. \r\nLa multitude d'univers parallèles et d'effets de miroir engendrent un &quot;délire circulaire&quot; vertigineux, une interrogation sur la relativité du temps et de l'espace. Dans quelle dimension sommes-nous ? Qui est ce &quot;je&quot; qui raconte l'invasion de la cité dans La Loterie de Babylone ? En mettant en vis-à-vis le Quichotte de Ménard et celui de Cervantès, lit-on la même chose ou bien la décision de redire suffit-elle à rendre la redite impossible ? \r\n

# Work on CSV

In [ ]:
nom_fichier_entree = 'PAL100.csv'

In [73]:
def traiter_csv_complet(fichier_entree='PAL100.csv', fichier_sortie='PAL_final.csv'):
    '''
    Parcourt l'intégralité du fichier CSV et remplit la colonne Résumé.
    Utilise la cascade de recherche : Livraddict -> Google Books -> Goodreads.

    Args:
        fichier_entree (str): Chemin du CSV source (ex: 'PAL.csv').
        fichier_sortie (str): Chemin pour sauvegarder les modifications.
    '''
    print(f"--- DÉMARRAGE DU TRAITEMENT COMPLET ---")
    
    # ETAPE 1 : Chargement du CSV 
    try:
        try:
            df = pd.read_csv(fichier_entree, sep=';', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(fichier_entree, sep=';', encoding='latin-1')
    except FileNotFoundError:
        print(f"Erreur : Le fichier {fichier_entree} est introuvable.")
        return

    # Préparation de la colonne Résumé
    if 'Résumé' not in df.columns:
        df['Résumé'] = ""

    total_lignes = len(df)
    print(f"Fichier chargé : {total_lignes} lignes à traiter.\n")

    # ETAPE 2 : Boucle sur toutes les lignes du DataFrame
    for index, row in df.iterrows():
        
        # Cas où il y a dejà un résumé - Sauter la ligne 
        if pd.notna(df.at[index, 'Résumé']) and len(str(df.at[index, 'Résumé'])) > 50:
            continue

        # ETAPE 2.A : RÉCUPÉRATION DES DONNÉES - Titre et Auteur 
        titre = str(df.iloc[index, 0])  
        auteur   = str(df.iloc[index, 2])  
        
        # Sécurité si le titre choisi est vide
        if not titre or titre == 'nan':
            titre = str(df.iloc[index, 1])

        print(f"[{index+1}/{total_lignes}] Traitement : {titre}...")

        # ETAPE 2.B : EXÉCUTION DE LA RECHERCHE
        try:
            resume_extrait = obtenir_resume_intelligent(titre, auteur)
            df.at[index, 'Résumé'] = resume_extrait
            
        except Exception as e:
            df.at[index, 'Résumé'] = f"Erreur système : {str(e)}"

        # ETAPE 2.C : SAUVEGARDE ET PAUSE 
        if (index + 1) % 10 == 0:
            df.to_csv(fichier_sortie, sep=';', index=False, encoding='utf-8-sig')
            print(f"   > Sauvegarde automatique effectuée ({index+1} lignes).")

        time.sleep(1.5)

    # ETAPE 3 : Sauvegarde finale
    df.to_csv(fichier_sortie, sep=';', index=False, encoding='utf-8-sig')
    print(f"\n--- TRAITEMENT TERMINÉ ---")
    print(f"Résultat disponible dans : {fichier_sortie}")


In [ ]:
traiter_csv_complet()

In [ ]:
def traiter_csv_precis(fichier_entree, fichier_sortie, titre_cible, auteur_cible):
    '''
    Recherche une ligne spécifique par son titre et son auteur pour y ajouter le résumé.
    
    Args:
        fichier_entree (str): Chemin du CSV source (ex: 'PAL.csv').
        fichier_sortie (str): Chemin pour sauvegarder les modifications.
        titre_cible (str): Le titre exact tel qu'il apparaît dans le CSV.
        auteur_cible (str): L'auteur exact tel qu'il apparaît dans le CSV.
    '''
    print(f"--- MISE À JOUR CIBLÉE ---")
    
    # ETAPE 1 : Chargement du CSV
    try:
        try:
            df = pd.read_csv(fichier_entree, sep=';', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(fichier_entree, sep=';', encoding='latin-1')
    except FileNotFoundError:
        print(f"Erreur : Le fichier {fichier_entree} est introuvable.")
        return

    # ETAPE 2 : Vérification de la colonne
    if 'Résumé' not in df.columns:
        df['Résumé'] = ""

    # ETAPE 3 : Identification de la ligne correspondante
    trouve = False
    for index, row in df.iterrows():
        titre_csv = str(df.iloc[index, 0]) 
        titre_vo_csv = str(df.iloc[index, 1]) 
        auteur_csv = str(df.iloc[index, 2])
        
        # Condition de correspondance 
        if (titre_cible == titre_csv or titre_cible == titre_vo_csv) and auteur_cible == auteur_csv:
            print(f"Ligne trouvée à l'index {index}. Lancement de la recherche...")

            # ETAPE 4 : Appel de la recherche intelligente et mise à jour de la cellule
            resume_extrait = obtenir_resume_intelligent(titre_cible, auteur_cible)
            
            df.at[index, 'Résumé'] = resume_extrait
            trouve = True
            break 

    if trouve:
        # ETAPE 5 : Sauvegarde avec l'encodage Excel
        df.to_csv(fichier_sortie, sep=';', index=False, encoding='utf-8-sig')
        print(f"Succès : Le résumé a été ajouté pour '{titre_cible}'.")
    else:
        print(f"Erreur : Impossible de trouver la combinaison '{titre_cible}' par '{auteur_cible}' dans le fichier.")


In [ ]:
#traiter_csv_precis('PAL_final.csv', 'PAL_final.csv', "Challenge Chicken Wings, Ou comment je suis devenu célèbre", "Tixier Jean-Christophe")

--- MISE À JOUR CIBLÉE ---
Ligne trouvée à l'index 94. Lancement de la recherche...
Recherche pour : Challenge Chicken Wings, Ou comment je suis devenu célèbre ...
Succès : Le résumé a été ajouté pour 'Challenge Chicken Wings, Ou comment je suis devenu célèbre'.
